# 04 · AI Functions on claim PDFs

One notebook, three layers. About 12 PDFs, roughly two minutes of AI Functions time.

| Layer | What | Tables |
|-------|------|--------|
| Bronze | `ai_parse_document` | `bronze_pdf_files`, `bronze_parsed_pdfs`, `bronze_parsed_text` |
| Silver | `ai_classify` + `ai_extract` | `silver_claim_documents`, `silver_claim_documents_typed` |
| Gold | metrics + Genie | `gold_clinic_invoices`, `gold_suspected_pdf_duplicates`, metric view |

**Planted duplicate:** Sparky · Pine Vet Clinic · `2026-08-03` · `CLM-PET-00044` / `CLM-PET-00045` (both Paid in `claims`).  
Bella is a waiting-period denial — not a duplicate.

**Prerequisite:** notebook `01_build_workshop_data` already wrote PDFs to `{PDF_DIR}`.


In [ ]:
%run ./00_setup


## Bronze — register + parse


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE bronze_pdf_files
COMMENT 'Binary PDF registry for pet-insurance claim packets'
AS
SELECT
  path,
  modificationTime,
  length,
  content,
  regexp_extract(path, '(CLM-PET-[0-9]+)', 1) AS claim_id,
  element_at(split(path, '/'), -1) AS file_name
FROM read_files('{PDF_DIR}', format => 'binaryFile')
WHERE path LIKE '%.pdf'
""")
display(spark.sql("SELECT claim_id, file_name, length FROM bronze_pdf_files ORDER BY claim_id, file_name"))
print("rows", spark.table("bronze_pdf_files").count())


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE bronze_parsed_pdfs
COMMENT 'ai_parse_document VARIANT output (v2.0) for pet-claim PDFs'
AS
SELECT
  claim_id,
  file_name,
  path,
  length,
  modificationTime,
  ai_parse_document(
    content,
    map('version', '2.0', 'descriptionElementTypes', '')
  ) AS parsed
FROM bronze_pdf_files
""")
# Do not display the VARIANT column — Spark Connect cannot materialize it in the notebook UI.
display(spark.sql("""
SELECT claim_id, file_name, length, typeof(parsed) AS parsed_type
FROM bronze_parsed_pdfs
ORDER BY claim_id, file_name
"""))


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE bronze_parsed_text
COMMENT 'Flattened parse text, one row per PDF'
AS
SELECT
  claim_id,
  file_name,
  path,
  concat_ws('\n', transform(
    from_json(to_json(parsed:document:elements), 'array<struct<content:string>>'),
    e -> e.content
  )) AS parsed_text
FROM bronze_parsed_pdfs
""")
display(spark.sql("""
SELECT claim_id, file_name, length(parsed_text) AS n_chars, substr(parsed_text, 1, 240) AS preview
FROM bronze_parsed_text
ORDER BY claim_id, file_name
"""))


## Silver — classify + extract


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE silver_claim_documents
COMMENT 'Classified + extracted pet-claim PDF fields'
AS
SELECT
  claim_id,
  file_name,
  path,
  parsed_text,
  ai_classify(
    substr(parsed_text, 1, 4000),
    '["fnol_claim_form", "clinic_invoice", "adjuster_report", "policy_declarations", "other"]'
  ) AS classified,
  ai_extract(
    substr(parsed_text, 1, 8000),
    '{
      "claim_number": {"type": "string", "description": "Claim number such as CLM-PET-00021"},
      "policy_number": {"type": "string", "description": "Policy number such as POL-88421"},
      "pet_name": {"type": "string", "description": "Name of the pet"},
      "species": {"type": "string", "description": "Dog or Cat"},
      "breed": {"type": "string", "description": "Breed of the pet"},
      "member_name": {"type": "string", "description": "Policyholder or clinic client name"},
      "clinic_name": {"type": "string", "description": "Veterinary clinic or hospital name"},
      "visit_date": {"type": "string", "description": "Visit date or date of loss as YYYY-MM-DD"},
      "invoice_number": {"type": "string", "description": "Clinic invoice number if this is an invoice"},
      "billed_amount": {"type": "number", "description": "Total billed amount as a number, no currency symbol"},
      "diagnosis": {"type": "string", "description": "Diagnosis or presenting complaint"},
      "recommended_decision": {"type": "string", "description": "Pay, Deny, or unknown"}
    }'
  ) AS extracted
FROM bronze_parsed_text
""")
print("silver_claim_documents ready")


In [ ]:
# Fix D: doc_type from classified:response[0]::string
# Fix E: fields from :response:<field>:value::string only (never the whole object)
spark.sql("""
CREATE OR REPLACE TABLE silver_claim_documents_typed
COMMENT 'Silver typed columns from ai_classify + ai_extract'
AS
SELECT
  claim_id AS packet_claim_id,
  file_name,
  path,
  classified:response[0]::string AS doc_type,
  extracted:response:claim_number:value::string AS claim_number,
  extracted:response:policy_number:value::string AS policy_number,
  extracted:response:pet_name:value::string AS pet_name,
  extracted:response:species:value::string AS species,
  extracted:response:breed:value::string AS breed,
  extracted:response:member_name:value::string AS member_name,
  extracted:response:clinic_name:value::string AS clinic_name,
  try_to_date(extracted:response:visit_date:value::string) AS visit_date,
  extracted:response:invoice_number:value::string AS invoice_number,
  try_cast(extracted:response:billed_amount:value::string AS DOUBLE) AS billed_amount,
  extracted:response:diagnosis:value::string AS diagnosis,
  extracted:response:recommended_decision:value::string AS recommended_decision
FROM silver_claim_documents
""")

display(spark.sql("""
SELECT
  packet_claim_id, doc_type, claim_number, pet_name, clinic_name,
  visit_date, invoice_number, billed_amount, diagnosis, recommended_decision
FROM silver_claim_documents_typed
ORDER BY packet_claim_id, doc_type
"""))


## Gold — invoices, duplicates, metric view


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE gold_clinic_invoices
COMMENT 'Invoice facts extracted from pet-claim PDFs'
AS
SELECT
  claim_number,
  policy_number,
  pet_name,
  species,
  breed,
  member_name,
  clinic_name,
  visit_date,
  invoice_number,
  billed_amount,
  diagnosis,
  recommended_decision,
  file_name
FROM silver_claim_documents_typed
WHERE doc_type = 'clinic_invoice'
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_claim_documents
COMMENT 'All typed PDF documents for Genie'
AS
SELECT * FROM silver_claim_documents_typed
""")

# Fix F: lower(trim(...)) so Pine Vet Clinic == PINE VET CLINIC
spark.sql("""
CREATE OR REPLACE TABLE gold_suspected_pdf_duplicates
COMMENT 'Invoices that share pet + clinic + visit_date (case-insensitive)'
AS
WITH d AS (
  SELECT
    lower(trim(pet_name)) AS pet_key,
    lower(trim(clinic_name)) AS clinic_key,
    visit_date,
    count(*) AS n_invoices
  FROM gold_clinic_invoices
  WHERE pet_name IS NOT NULL AND clinic_name IS NOT NULL AND visit_date IS NOT NULL
  GROUP BY 1, 2, 3
  HAVING count(*) > 1
)
SELECT i.*, d.n_invoices
FROM gold_clinic_invoices i
JOIN d
  ON lower(trim(i.pet_name)) = d.pet_key
 AND lower(trim(i.clinic_name)) = d.clinic_key
 AND i.visit_date = d.visit_date
""")

display(spark.sql("SELECT * FROM gold_clinic_invoices ORDER BY visit_date, claim_number"))
display(spark.sql("SELECT * FROM gold_suspected_pdf_duplicates ORDER BY visit_date, claim_number"))


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold_pdf_invoice_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Metric view over PDF-extracted clinic invoices'
AS $$
version: 0.1
source: {CATALOG}.{SCHEMA}.gold_clinic_invoices
filter: billed_amount IS NOT NULL
dimensions:
  - name: Clinic
    expr: clinic_name
  - name: Diagnosis
    expr: diagnosis
  - name: Pet
    expr: pet_name
  - name: Visit date
    expr: visit_date
  - name: Claim
    expr: claim_number
measures:
  - name: Invoice count
    expr: COUNT(1)
  - name: Total billed
    expr: SUM(billed_amount)
  - name: Average billed
    expr: AVG(billed_amount)
$$
""")
display(spark.sql("""
SELECT
  `Clinic` AS clinic,
  MEASURE(`Invoice count`) AS invoice_count,
  MEASURE(`Total billed`) AS total_billed,
  MEASURE(`Average billed`) AS avg_billed
FROM gold_pdf_invoice_metrics
GROUP BY `Clinic`
ORDER BY total_billed DESC
"""))


In [ ]:
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN pet_name COMMENT 'Pet name extracted from the invoice PDF'
""")
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN clinic_name COMMENT 'Veterinary clinic extracted from the invoice PDF'
""")
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN visit_date COMMENT 'Visit date. Duplicate key is lower(trim(pet_name)) + lower(trim(clinic_name)) + visit_date'
""")
spark.sql("""
ALTER TABLE gold_suspected_pdf_duplicates SET TBLPROPERTIES (
  'comment' = 'Sparky at Pine Vet Clinic on 2026-08-03 is the planted pair (CLM-PET-00044 / 00045).'
)
""")
print("comments applied")


## Check — exactly one PDF duplicate pair

Expect **Sparky · Pine Vet Clinic · 2026-08-03 · CLM-PET-00044 / CLM-PET-00045**.


In [ ]:
display(spark.sql("""
SELECT
  lower(trim(pet_name)) AS pet,
  lower(trim(clinic_name)) AS clinic,
  visit_date,
  collect_list(claim_number) AS claims,
  collect_list(invoice_number) AS invoices,
  count(*) AS n
FROM gold_clinic_invoices
GROUP BY 1, 2, 3
HAVING count(*) > 1
"""))


## Genie on gold tables

**New → Genie space.** Add:

- `{catalog}.{schema}.gold_clinic_invoices`
- `{catalog}.{schema}.gold_claim_documents`
- `{catalog}.{schema}.gold_suspected_pdf_duplicates`
- `{catalog}.{schema}.gold_pdf_invoice_metrics`


In [ ]:
instructions = f"""
You are a pet-insurance claims analyst for a workshop. Answer only from the attached Unity Catalog objects in {CATALOG}.{SCHEMA}.

Tables
- gold_clinic_invoices: one row per clinic invoice PDF. Grain is invoice_number.
- gold_claim_documents: every parsed PDF (FNOL, invoice, adjuster, policy) with doc_type.
- gold_suspected_pdf_duplicates: invoices that share pet_name + clinic_name + visit_date (case-insensitive).
- gold_pdf_invoice_metrics: metric view. Use MEASURE(`Invoice count`), MEASURE(`Total billed`), MEASURE(`Average billed`).

Duplicate definition
Same pet_name, clinic_name, and visit_date on more than one invoice. Invoice number is not the key.
CLM-PET-00044 and CLM-PET-00045 are the planted pair (Sparky, Pine Vet Clinic, 2026-08-03).
CLM-PET-00088 is a waiting-period denial, not a duplicate.

Rules
- If a question cannot be answered from these objects, say so.
- Prefer gold_suspected_pdf_duplicates when asked about duplicates.
- Money is USD.
""".strip()
print(instructions)
print("\nBenchmark questions:")
for i, q in enumerate([
    "How many clinic invoices did we extract from PDFs, and what is total billed?",
    "Which invoices share the same pet, clinic, and visit date?",
    "What did Pine Vet Clinic bill for Sparky, and how many claim numbers are involved?",
    "Is Bella's Maple Emergency Vets invoice a duplicate?",
    "Break down total billed by clinic using the metric view.",
], 1):
    print(f"  {i}. {q}")


## Your turn — Lab D complete when

- [ ] `gold_suspected_pdf_duplicates` shows exactly one pair: Sparky / Pine Vet / 2026-08-03
- [ ] Those claim numbers also exist as **Paid** in the synthetic `claims` table (`CLM-PET-00044`, `CLM-PET-00045`)
- [ ] Genie on the gold tables answers the Sparky question without inventing a new duplicate key

Optional stretch: wrap bronze → silver → gold as a Lakeflow Spark Declarative Pipeline.
